In [1]:
# Automatically reloads all code changes in python modules when changes are saved
%load_ext autoreload
%autoreload 2

In [2]:
## Initialize logger for whole project
## Also tracks output of YOLO ML training to this log file

from logging_config import setup_logging
from ultralytics import YOLO

# LOG_FILE = 'flower_classifier.log'
# logger = setup_logging(log_file=str(LOG_FILE))
logger = setup_logging()
logger.debug(f"Logging setup successful.")

2026-04-06 11:01:26,283 - flower_classifier - INFO - Writing log to: '/home/jdc6281/Coding/thesis/flower-classifier/process/flower_classifier.log'


In [13]:
## Sets up environment variables with kaggle
## MUST HAVE .env FILE WITH FILLED IN KAGGLE LOGIN INFO EVEN IF USE_KAGGLE IS FALSE
from dotenv import load_dotenv
from pathlib import Path
import os

file_path = ".env"
if not os.path.isfile(file_path):
    logger.error("'.env' file not found in this directory. Please make .env file with KAGGLE_USERNAME and KAGGLE_KEY. See README.md for more information.")
else:
    logger.info(".env file found! Importing configs.")

# Extract environment variables from .env file
load_dotenv()
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')
GOOGLE_ZIP_ID = os.getenv('GOOGLE_ZIP_ID')

# Resolve project root directory
path = Path(__name__).resolve().parent
while path.name != "flower-classifier" and path != path.parent:
    path = path.parent
logger.info(f"Path to project parent directory: '{path}'")
PROJECT_ROOT = path

2026-04-06 11:47:24,287 - flower_classifier - INFO - .env file found! Importing configs.
2026-04-06 11:47:24,297 - flower_classifier - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'


In [4]:
#from get_dataset import get_dataset
#from ingest import ingest
#from config_IO import save_json_config, load_json_config, make_YOLO26_config
#from system_info import print_system_info, verify_venv
#from view_file import view_image, view_text

import get_dataset
import ingest
import config_IO
import system_info
import view_file

from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch

2026-04-06 11:01:32,544 - flower_classifier.get_dataset - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'


In [5]:
system_info.verify_venv()

2026-04-06 11:01:37,568 - flower_classifier.system_info - INFO - Jupyter is running in venv!


True

In [7]:
USE_KAGGLE = False

do_unzip = get_dataset.get_dataset(from_kaggle=USE_KAGGLE, kaggle_username=KAGGLE_USERNAME, kaggle_key=KAGGLE_KEY, google_zip_id=GOOGLE_ZIP_ID)

if do_unzip:
    get_dataset.unzip_dataset()

2026-04-06 11:04:57,228 - flower_classifier.get_dataset - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'
2026-04-06 11:04:58,902 - flower_classifier.get_dataset - INFO - Retrieving dataset from google drive...
2026-04-06 11:04:58,903 - flower_classifier.get_dataset - INFO - Downloading compressed dataset via Google Drive to '/home/jdc6281/Coding/thesis/flower-classifier/data/compressed/tomato-flower-3-class.zip'


Downloading...
From (original): https://drive.google.com/uc?id=1GWKMcoQ52PxBDwIWmCX1JEPnmucnyNl-
From (redirected): https://drive.google.com/uc?id=1GWKMcoQ52PxBDwIWmCX1JEPnmucnyNl-&confirm=t&uuid=f6f35b44-baad-419d-85c8-d752f92535bb
To: /home/jdc6281/Coding/thesis/flower-classifier/data/compressed/tomato-flower-3-class.zip
100%|████████████████████████████████████████| 711M/711M [00:18<00:00, 37.9MB/s]


2026-04-06 11:05:26,073 - flower_classifier.get_dataset - INFO - Download complete!
2026-04-06 11:05:26,074 - flower_classifier.get_dataset - INFO - Extracting files to directory: '/home/jdc6281/Coding/thesis/flower-classifier/data/data_set'
2026-04-06 11:16:08,354 - flower_classifier.get_dataset - INFO - Files extracted successfully!


In [8]:
ingest.ingest()

2026-04-06 11:16:23,481 - flower_classifier.ingest - INFO - Total source images: 3636
2026-04-06 11:16:23,483 - flower_classifier.ingest - INFO - Validation images:   1818
2026-04-06 11:16:23,483 - flower_classifier.ingest - INFO - Test images:         1818
2026-04-06 11:22:25,296 - flower_classifier.ingest - INFO - DONE! Split complete.
2026-04-06 11:22:28,212 - flower_classifier.ingest - INFO -              
 TRAIN:             
   images = 14545             
   labels = 14545             
 
2026-04-06 11:22:28,316 - flower_classifier.ingest - INFO -              
 VAL:             
   images = 1818             
   labels = 1818             
 
2026-04-06 11:22:28,416 - flower_classifier.ingest - INFO -              
 TEST:             
   images = 1818             
   labels = 1818             
 
2026-04-06 11:22:28,436 - flower_classifier.ingest - INFO - DONE! Wrote data.yaml to: /home/jdc6281/Coding/thesis/flower-classifier/config/data.yaml
2026-04-06 11:22:28,446 - flower_classifi

In [15]:
config_IO.make_YOLO26_config()

2026-04-06 12:16:12,122 - flower_classifier.config_IO - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'
2026-04-06 12:16:12,125 - flower_classifier.config_IO - INFO - Creating default config set at location '/home/jdc6281/Coding/thesis/flower-classifier/config/YOLO26_config.json'
2026-04-06 12:16:12,127 - flower_classifier.config_IO - INFO - Default config set: 
{
    "model_cfg": "yolo26n.yaml",
    "data_yaml": "/home/jdc6281/Coding/thesis/flower-classifier/config/data.yaml",
    "project_dir": "/home/jdc6281/Coding/thesis/flower-classifier/runs",
    "run_name": "/home/jdc6281/Coding/thesis/flower-classifier/tomato_flower_yolo26_from_scratch",
    "train_params": {
        "epochs": 100,
        "imgsz": 640,
        "batch": -1,
        "device": -1,
        "workers": 8,
        "patience": 30,
        "optimizer": "auto",
        "lr0": 0.01,
        "lrf": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_ep

In [10]:
system_info.print_system_info()

2026-04-06 11:22:43,614 - flower_classifier.system_info - INFO -          
 torch 2.11.0+cu130         
 torchvision 0.26.0+cu130         
 ultralytics 8.4.33         
 cv2 4.13.0         
 captum 0.7.0         
 CUDA available: True         
 Python 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]         
 Executable /home/jdc6281/Coding/thesis/.venv/bin/python         
 Platform Linux-5.15.0-173-generic-x86_64-with-glibc2.35         



In [17]:
## Load parameters from specified file location

file_location =  str(PROJECT_ROOT / "config/YOLO26_config.json")
TRAIN_CONFIG = config_IO.load_json_config(file_location)

2026-04-06 12:52:33,581 - flower_classifier.config_IO - INFO - DONE! Loaded config from /home/jdc6281/Coding/thesis/flower-classifier/config/YOLO26_config.json
2026-04-06 12:52:33,582 - flower_classifier.config_IO - INFO - Loaded config set: 
{
    "model_cfg": "yolo26n.yaml",
    "data_yaml": "/home/jdc6281/Coding/thesis/flower-classifier/config/data.yaml",
    "project_dir": "/home/jdc6281/Coding/thesis/flower-classifier/runs",
    "run_name": "/home/jdc6281/Coding/thesis/flower-classifier/tomato_flower_yolo26_from_scratch",
    "train_params": {
        "epochs": 100,
        "imgsz": 640,
        "batch": -1,
        "device": -1,
        "workers": 8,
        "patience": 30,
        "optimizer": "auto",
        "lr0": 0.01,
        "lrf": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_epochs": 3.0,
        "warmup_momentum": 0.8,
        "warmup_bias_lr": 0.1,
        "box": 7.5,
        "cls": 0.5,
        "dfl": 1.5,
        "close_mosaic": 10,


In [18]:
## Manual overrides

## for example:
# cfg["run_name"] = "tomato_flower_yolo26_trial_02"
# cfg["train_params"]["epochs"] = 150

## We change the configs within the 'cfg' object as opposed to directly on MODEL_CFG or other imported vars.
## This is so, whenever we save the configs, they can be exported back to the JSON config file if we wish.

In [19]:
## CPU/GPU Usage Config

CPU_COUNT = multiprocessing.cpu_count()
DATALOADER_WORKERS = TRAIN_CONFIG["train_params"]["workers"]
CPU_THREADS_PER_PROC = max(1, CPU_COUNT // max(1, DATALOADER_WORKERS))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)

torch.set_num_threads(CPU_THREADS_PER_PROC)

print(f"CPU count: {CPU_COUNT}")
print(f"DataLoader workers: {DATALOADER_WORKERS}")
print(f"CPU threads per process: {CPU_THREADS_PER_PROC}")
print(f"CUDA available: {torch.cuda.is_available()}")

CPU count: 16
DataLoader workers: 8
CPU threads per process: 2
CUDA available: True


In [21]:
## PREFLIGHT CHECKS
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.


# ==========================
# USER CONFIG (from your setup)
# ==========================
DATA_YAML = PROJECT_ROOT / "config/data.yaml"
DATASET_ROOT = PROJECT_ROOT / "data/data_set/tomato-flower"
PROJECT_DIR = PROJECT_ROOT / "runs"
RUN_NAME = "tomato_flower_yolo26_from_scratch"
MODEL_CFG = "yolo26n.yaml"

TRAIN_PARAMS = TRAIN_CONFIG["train_params"]

# ==========================
# CPU THREAD CONTROL (GPU TRAINING SAFE)
# ==========================
CPU_COUNT = multiprocessing.cpu_count()
WORKERS = TRAIN_PARAMS.get("workers", 8)

CPU_THREADS_PER_PROC = max(1, CPU_COUNT // max(1, WORKERS))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)

torch.set_num_threads(CPU_THREADS_PER_PROC)

logger.info(
    f"\n=== CPU THREAD CONFIG ==="
    f"\n CPU count              : {CPU_COUNT}"
    f"\n DataLoader workers     : {WORKERS}"
    f"\n Threads per process    : {CPU_THREADS_PER_PROC}"
    f"\n torch num threads      : {torch.get_num_threads()}"
    f"\n"
)

# ==========================
# SYSTEM INFO
# ==========================
logger.info(
    f"\n=== SYSTEM INFO ==="
    f"\n Python executable : {sys.executable}"
    f"\n Python version    : {sys.version.splitlines()[0]}"
    f"\n Platform          : {platform.platform()}"
    f"\n Machine           : {platform.machine()}"
    f"\n Processor         : {platform.processor()}"
    f"\n CPU count         : {CPU_COUNT}"
    f"\n Working dir       : {Path.cwd()}"
    f"\n"
)

# ==========================
# TRAIN CONFIG SUMMARY
# ==========================
logger.info(
    f"\n=== TRAIN CONFIG SUMMARY ==="
    f"\n model_cfg   : {TRAIN_CONFIG['model_cfg']}"
    f"\n data_yaml   : {TRAIN_CONFIG['data_yaml']}"
    f"\n project_dir : {TRAIN_CONFIG['project_dir']}"
    f"\n run_name    : {TRAIN_CONFIG['run_name']}"
    f"\n epochs      : {TRAIN_PARAMS['epochs']}"
    f"\n imgsz       : {TRAIN_PARAMS['imgsz']}"
    f"\n batch       : {TRAIN_PARAMS['batch']}"
    f"\n device      : {TRAIN_PARAMS['device']}"
    f"\n workers     : {TRAIN_PARAMS['workers']}"
    f"\n amp         : {TRAIN_PARAMS['amp']}"
    f"\n optimizer   : {TRAIN_PARAMS['optimizer']}"
    f"\n pretrained  : {TRAIN_PARAMS['pretrained']}"
    f"\n"
)

# ==========================
# TORCH / CUDA INFO
# ==========================
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

logger.info(
    f"\n=== TORCH INFO ==="
    f"\n torch version       : {torch.__version__}"
    f"\n CUDA available      : {cuda_available}"
    f"\n CUDA device count   : {torch.cuda.device_count() if cuda_available else 0}"
    f"\n MPS available       : {mps_available}"
    f"\n"
)

if cuda_available:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_mem_gb = props.total_memory / (1024 ** 3)

        logger.info(
            f"\nCUDA DEVICE {i}:"
            f"\n name              = {torch.cuda.get_device_name(i)}"
            f"\n capability        = {props.major}.{props.minor}"
            f"\n total_memory_gb   = {total_mem_gb:.2f}"
            f"\n multiprocessors   = {props.multi_processor_count}"
        )
else:
    logger.warning("CUDA is NOT available. Training will fall back to CPU.")

# ==========================
# CUDA SMOKE TEST
# ==========================
logger.info(f"Testing CUDA usage...")
if cuda_available:
    try:
        x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
        logger.info(f"CUDA smoke test PASSED (tensor device = {x.device})")
        del x
        torch.cuda.empty_cache()
    except Exception as e:
        logger.exception(f"CUDA detected but failed runtime test: {e}")
        raise RuntimeError(f"CUDA detected but failed runtime test: {e}")

# ==========================
# ENV VARS
# ==========================
cuda_visible_devices = os.environ.get('CUDA_VISIBLE_DEVICES')
logger.info(
    f"\n=== IMPORTANT ENV VARS ==="
    f"\n OMP_NUM_THREADS        = {os.environ.get('OMP_NUM_THREADS')}"
    f"\n MKL_NUM_THREADS        = {os.environ.get('MKL_NUM_THREADS')}"
    f"\n NUMEXPR_NUM_THREADS    = {os.environ.get('NUMEXPR_NUM_THREADS')}"
    f"\n OPENBLAS_NUM_THREADS   = {os.environ.get('OPENBLAS_NUM_THREADS')}"
    f"\n CUDA_VISIBLE_DEVICES   = {cuda_visible_devices if cuda_visible_devices is not None else '<Not restricted>'}"
    f"\n ULTRALYTICS_CONFIG_DIR = {os.environ.get('ULTRALYTICS_CONFIG_DIR')}"
    f"\n"
)

if os.environ.get("CUDA_VISIBLE_DEVICES") is not None:
    logger.info(
        f"CUDA_VISIBLE_DEVICES is set to {os.environ.get('CUDA_VISIBLE_DEVICES')} "
        "→ This may restrict visible GPUs."
    )

# ==========================
# DEVICE VALIDATION
# ==========================
requested_device = TRAIN_PARAMS["device"]

if requested_device != "cpu" and not cuda_available:
    raise RuntimeError(
        f"Config requests GPU (device={requested_device}) but CUDA is unavailable."
    )

if requested_device == -1:
    logger.info("device=-1 → auto-selecting most idle GPU.")

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    logger.info(
        f"\n{label}:"
        f"\n path    = {path}"
        f"\n exists  = {path.exists()}"
        f"\n is_dir  = {path.is_dir()}"
        f"\n is_file = {path.is_file()}"
    )

# ==========================
# PATH CHECKS
# ==========================
logger.info("\n=== PATH CHECKS ===")

print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)

    img_count = count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    lbl_count = count_files(lbl_dir, {".txt"})

    logger.info(
        f"{split.upper()} COUNTS:"
        f"\n image files = {img_count}"
        f"\n label files = {lbl_count}"
    )

# ==========================
# YAML VALIDATION
# ==========================
logger.info("\n=== YAML CONTENTS ===")

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

logger.info(json.dumps(data_cfg, indent=2))

logger.info("\n=== YAML PATH VALIDATION ===")

for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    logger.info(
        f"{key}: {p}"
        f"\n exists = {p.exists()}"
        f"\n is_dir = {p.is_dir()}"
    )

logger.info("\n=== PREFLIGHT CHECK COMPLETE ===\n")

2026-04-06 13:07:19,614 - flower_classifier - INFO - 
=== CPU THREAD CONFIG ===
 CPU count              : 16
 DataLoader workers     : 8
 Threads per process    : 2
 torch num threads      : 2

2026-04-06 13:07:19,617 - flower_classifier - INFO - 
=== SYSTEM INFO ===
 Python executable : /home/jdc6281/Coding/thesis/.venv/bin/python
 Python version    : 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
 Platform          : Linux-5.15.0-173-generic-x86_64-with-glibc2.35
 Machine           : x86_64
 Processor         : x86_64
 CPU count         : 16
 Working dir       : /home/jdc6281/Coding/thesis/flower-classifier/process

2026-04-06 13:07:19,618 - flower_classifier - INFO - 
=== TRAIN CONFIG SUMMARY ===
 model_cfg   : yolo26n.yaml
 data_yaml   : /home/jdc6281/Coding/thesis/flower-classifier/config/data.yaml
 project_dir : /home/jdc6281/Coding/thesis/flower-classifier/runs
 run_name    : /home/jdc6281/Coding/thesis/flower-classifier/tomato_flower_yolo26_from_scratch
 epochs      : 100
 

In [22]:
## Verify YOLO26 is accessible and usable

try:
    YOLO("yolo26n.yaml")
    logger.info("DONE! Model config recognized")
except Exception as e:
    logger.info("ERROR! Model config NOT recognized")
    logger.info(e)

2026-04-06 13:14:15,807 - flower_classifier - INFO - DONE! Model config recognized


In [ ]:
## Training the YOLO26 model

if RUN_NUM is not None:
    RUN_NUM += 1
else:
    RUN_NUM = 1

try:
    logger.info(f"Beginning training run {RUN_NUM}")
    
    model = YOLO(cfg["model_cfg"])
    
    results = model.train(
        data=cfg["data_yaml"],
        project=cfg["project_dir"],
        name=cfg["run_name"],
        **cfg["train_params"]
    )

    logger.info(f"DONE! Training run {RUN_NUM} complete!")

    
    ## Save current config values back to specified config json file.
    CONFIG_PATH = str(PROJECT_ROOT / f"runs/YOLO26_config_r{RUN_NUM}.json")
    logger.info(f"Saving configs for run {RUN_NUM} to '{CONFIG_PATH}'")
    
    config_IO.save_json_config(cfg, CONFIG_PATH)
    logger.info("Configs saved successfully!")
    
except Exception as e:
    logger.exception(f"ERROR while training model: \n{e}")


logger.info("Training complete!")